In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def importData():
    csv_path =  "../data/processed/mergedData.csv"
    
    data = pd.read_csv(csv_path)
    print("Dataset Length:", len(data))
    print("Dataset Shape:", data.shape)
    print("Dataset Head:\n", data.head())
    return data

def splitDataset(data):
    features = data.drop(columns=["prognosis"])
    disease_labels = data["prognosis"]
    X_train, X_test, y_train, y_test = train_test_split(
        features,
        disease_labels,
        test_size=0.2,
        random_state=100,
        stratify=disease_labels,
    )
    
    
    plt.figure(figsize=(14, 6))
    disease_counts = data['prognosis'].value_counts()
    sns.barplot(x=disease_counts.values, y=disease_counts.index, palette='viridis')
    plt.title('Number of Samples per Disease', fontsize=14)
    plt.xlabel('Count')
    plt.ylabel('Disease')
    plt.tight_layout()
    plt.show()


    symptom_counts = X_train.sum().sort_values(ascending=False).head(20)

    plt.figure(figsize=(14, 6))
    sns.barplot(x=symptom_counts.values, y=symptom_counts.index, palette='magma')
    plt.title('Top 20 Most Frequently Occurring Symptoms', fontsize=14)
    plt.xlabel('Total Occurrences Across All Patients')
    plt.ylabel('Symptom')
    plt.tight_layout()
    plt.show()


    data_copy = data.copy()
    symptom_cols = X_train.columns.tolist()

    avg_symptoms = data_copy.groupby('prognosis')[symptom_cols].sum().sum(axis=1).sort_values(ascending=False)

    plt.figure(figsize=(14, 6))
    sns.barplot(x=avg_symptoms.values, y=avg_symptoms.index, palette='coolwarm')
    plt.title('Total Active Symptoms per Disease (across all samples)', fontsize=14)
    plt.xlabel('Total Symptom Occurrences')
    plt.ylabel('Disease')
    plt.tight_layout()
    plt.show()

    top_20_symptoms = X_train.sum().sort_values(ascending=False).head(20).index.tolist()

    heatmap_data = data_copy.groupby('prognosis')[top_20_symptoms].mean()

    plt.figure(figsize=(18, 12))
    sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.5, annot=False)
    plt.title('Disease vs Top 20 Symptoms (Mean Presence)', fontsize=14)
    plt.xlabel('Symptom')
    plt.ylabel('Disease')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return X_train, X_test, y_train, y_test


def train_using_kmeans(features, disease_labels, n_clusters=None):
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    if n_clusters is None:
        n_clusters = disease_labels.nunique()

    kmeans = KMeans(n_clusters=n_clusters, random_state=100, n_init=10)
    cluster_labels = kmeans.fit_predict(features_scaled)
    cluster_disease = (
        pd.DataFrame(
            {"cluster": cluster_labels, "disease": disease_labels.to_numpy()}
        )
        .groupby("cluster")["disease"]
        .agg(lambda labels: labels.mode().iloc[0])
        .to_dict()
    )

    return kmeans, scaler, cluster_disease, features_scaled, cluster_labels


def plot_kmeans_clusters_tsne(features_scaled, cluster_labels, kmeans, cluster_disease):
    """
    Uses t-SNE for non-linear dimensionality reduction and projects centroids cleanly into the 2D space.
    """
    n_samples = features_scaled.shape[0]

    all_points = np.vstack([features_scaled, kmeans.cluster_centers_])

    tsne = TSNE(n_components=2, perplexity=min(30, n_samples - 1), random_state=100)
    tsne_all = tsne.fit_transform(all_points)

    tsne_features = tsne_all[:n_samples]
    tsne_centroids = tsne_all[n_samples:]

    mapped_labels = np.array([f"Cluster {c}: {cluster_disease[c]}" for c in cluster_labels])

    plt.figure(figsize=(14, 8))
    sns.scatterplot(
        x=tsne_features[:, 0],
        y=tsne_features[:, 1],
        hue=mapped_labels,
        palette="tab20",
        alpha=0.6,
        s=80,
    )
    plt.scatter(
        tsne_centroids[:, 0],
        tsne_centroids[:, 1],
        c="red",
        marker="X",
        s=150,
        label="Cluster Centroids",
        edgecolor="black",
        zorder=5,
    )
    for idx, (x_pos, y_pos) in enumerate(tsne_centroids):
        plt.text(
            x_pos,
            y_pos,
            f"  {idx}",
            fontsize=9,
            weight="bold",
            color="black",
            zorder=6,
        )
    plt.title("K-Means Clustering Visualization (2D t-SNE Projection)", fontsize=14)
    plt.xlabel("t-SNE Dimension 1", fontsize=12)
    plt.ylabel("t-SNE Dimension 2", fontsize=12)

    # Place the legend horizontally below the plot
    plt.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=4,
        borderaxespad=0.0,
        fontsize=9
    )

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


def prediction(features, kmeans, scaler, cluster_disease):
    features_scaled = scaler.transform(features)
    cluster_labels = kmeans.predict(features_scaled)
    predicted_diseases = pd.Series(cluster_labels, index=features.index).map(
        cluster_disease
    )
    if predicted_diseases.isna().any():
        raise ValueError(
            "Some predicted clusters do not have a corresponding disease label."
        )
    print("\nPredicted Diseases:\n", predicted_diseases.to_numpy())
    return predicted_diseases


def cal_accuracy(y_test, y_pred):
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    accuracy = accuracy_score(y_test, y_pred)
    print("\nAccuracy:", accuracy * 100)
    print(
        "\nClassification Report:\n",
        classification_report(y_test, y_pred, zero_division=0),
    )
    return accuracy


data = importData()
X_train, X_test, y_train, y_test = splitDataset(data)

print("\n Training using K-Means...")
kmeans, scaler, cluster_disease, X_train_scaled, train_cluster_labels = train_using_kmeans(
    X_train, y_train
)

plot_kmeans_clusters_tsne(X_train_scaled, train_cluster_labels, kmeans, cluster_disease)
print("\n Training using K-Means")
y_pred = prediction(X_test, kmeans, scaler, cluster_disease)
cal_accuracy(y_test, y_pred)



ModuleNotFoundError: No module named 'sklearn'